# Week 2 · Lab 02
## Resilient API Harvester + SQL Extractor — *with Provenance*

> **AI Engineering Academy** · Gamut Technology Services

Production data extraction has to survive the real world — **auth**, **rate limits**,
**transient failures** — and it has to be **defensible**: for every page you harvest you
should be able to answer *where did this come from, when, and is it intact?* That's
**provenance**, and it's what makes a dataset you can trust to train or ground an LLM.

This lab builds a resilient, authenticated **harvester** that paginates an API, retries
intelligently, and records a **provenance manifest** (source, params, timestamp, SHA-256
checksum, byte size) for every page — plus an **incremental** mode driven by a
**high-watermark**. Then you extract the same data via **SQL** and validate it.

> **Continues Lab 01.** Same synthetic Cordwell database; now the API requires a
> **Bearer token** and enforces a **rate limit** (real `429` + `Retry-After`).

### ⚙️ No external internet required
Setup builds `cordwell.db` and starts the local **FastAPI** (`harvest_api.py`) on
`http://127.0.0.1:8000`. See `API_REFERENCE.md` and `cordwell_data_dictionary.md`.

### Learning objectives
1. Build a **resilient GET** with Bearer auth, exponential backoff **+ jitter**, and `Retry-After` handling; treat `401/403` as non-retryable.
2. Harvest with **cursor pagination**, persisting **raw snapshots** + a **provenance manifest** (checksum, bytes, timestamp).
3. Run **incremental harvests** driven by a **high-watermark**.
4. Normalize to **partitioned Parquet**; extract the same slice via **parameterized SQL** with **chunked** streaming writes.
5. (Stretch) Enforce a **data contract** (pandera), make runs **idempotent** (upsert), and bound retries with a **sleep budget**.

### Time budget — ~135 min
| Segment | Time |
|---|---|
| Setup | 8 min |
| **A.** Resilient harvester + provenance + watermark | 60 min |
| **B.** SQL extractor (params, chunked Parquet) | 30 min |
| **C.** Stretch: contract · idempotency · retry budget | 30 min |
| Wrap-up | 7 min |

### Files beside this notebook
- `build_cordwell_db.py`, `harvest_api.py` — run/started for you in setup.
- `API_REFERENCE.md`, `cordwell_data_dictionary.md` — reference docs.


In [ ]:
%pip install -r requirements.txt

In [ ]:
# --- Setup: build the DB, start the AUTHENTICATED, rate-limited API ---------
# No external network calls. A generator builds the synthetic Cordwell Home &
# Hardware SQLite database (same as Lab 01), and a small FastAPI app serves it
# behind Bearer-token auth + a rate limiter that returns real 429 + Retry-After.
import os, json, hashlib, time, random, shutil
import datetime as dt
import sqlite3
from pathlib import Path
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import build_cordwell_db            # the generator (shipped beside this notebook)
from harvest_api import start_server  # the local auth API (shipped beside this notebook)

# 1) Build the database if missing (seeded / reproducible).
DB_PATH = "cordwell.db"
if not Path(DB_PATH).exists():
    print("Building database:", build_cordwell_db.build(DB_PATH, n_orders=10_000, seed=2025))
else:
    print("Database present:", DB_PATH)

# 2) Configure auth + start the API on a background thread.
TOKEN = "cordwell-dev-token"
os.environ["CORDWELL_DB"] = DB_PATH
os.environ["HARVEST_TOKEN"] = TOKEN
BASE_URL = "http://127.0.0.1:8000"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}   # sent on every /v1/* request

server, _thread = start_server(port=8000)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("Local API ready:", requests.get(f"{BASE_URL}/health", timeout=2).json())

def check(label, predicate):
    """Soft self-check: prints PASS/FAIL, never raises."""
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — Resilient Harvester with Provenance

### A1 — Scaffolding  *(guided)*
Set up the artifact layout the harvest writes to. Clearing it first makes every run
reproducible (deterministic snapshot/manifest counts).


In [ ]:
ARTIFACTS = Path("artifacts")
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)            # fresh start -> reproducible counts
RAW_DIR = ARTIFACTS / "raw"
PARQUET_DIR = ARTIFACTS / "parquet"
MANIFEST = ARTIFACTS / "manifest.jsonl"
for d in (RAW_DIR, PARQUET_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("artifact dirs ready under", ARTIFACTS.resolve().name + "/")
print("auth header:", {k: v[:20] + "..." for k, v in HEADERS.items()})

### A2 — A resilient GET (auth · backoff+jitter · `Retry-After`)
Implement `resilient_get(url, params=None, headers=None, max_retries=6)`:

- issue a `GET` with a **timeout**;
- **200** → return the `Response`;
- **429** → wait `Retry-After` seconds if present (else the current backoff), then retry;
- **500/502/503/504** → wait the current backoff, then retry;
- **any other status** (incl. **401/403**) → raise `HarvestError` immediately (not retryable);
- `Timeout`/`ConnectionError` → wait, then retry;
- add **jitter** to every sleep and **cap** the backoff; after `max_retries`, raise `HarvestError`.


💡 **Hint.** `backoff = 0.5`. Loop `for attempt in range(1, max_retries+1)`. Wrap the
`requests.get` in `try/except (requests.Timeout, requests.ConnectionError)`. For a retryable
status compute `delay`, then `time.sleep(delay + random.uniform(0, 0.25*delay))` and
`backoff = min(backoff*2, 8.0)`. Reset the server's counters with
`requests.post(f"{BASE_URL}/admin/reset")` before testing the flaky endpoints.


In [ ]:
class HarvestError(Exception):
    pass

def resilient_get(url, params=None, headers=None, max_retries=6):
    backoff = 0.5
    headers = headers or {}
    # TODO: loop up to max_retries:
    #   - GET (catch Timeout/ConnectionError -> sleep + backoff, continue)
    #   - 200 -> return resp
    #   - 429 -> delay = float(Retry-After) if present else backoff
    #   - 500/502/503/504 -> delay = backoff
    #   - else -> raise HarvestError (non-retryable, e.g. 401/403)
    #   - sleep(delay + jitter); backoff = min(backoff*2, 8.0)
    raise NotImplementedError

In [ ]:
requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_u = resilient_get(f"{BASE_URL}/v1/unreliable", {"key": "chkA2", "fail_times": 2}, HEADERS).json()
check("A2: recovered from 503s via backoff (attempts == 3)", lambda: _u["attempts"] == 3)

requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_r = resilient_get(f"{BASE_URL}/v1/rate-limited", {"key": "chkA2rl"}, HEADERS).json()
check("A2: honored Retry-After on 429 (attempts == 2)", lambda: _r["attempts"] == 2)

def _expect_auth_error():
    try:
        resilient_get(f"{BASE_URL}/v1/orders", {"limit": 1}, headers={})  # no token
        return False
    except HarvestError:
        return True
check("A2: missing token (401) raises immediately, not retried", _expect_auth_error)

### A3 — Harvest with cursor pagination + provenance
Now the heart of the lab. Implement two functions:

1. `save_snapshot(payload_bytes, meta)` — write the raw page bytes to
   `RAW_DIR/orders_<UTC-timestamp>_<page>.json`, and **append one line** to `MANIFEST`
   (JSONL) recording `meta` plus `timestamp`, `sha256` (of the bytes), and `bytes` (length).
2. `harvest_orders(base, headers, page_size=1000, **filters)` — walk the cursor pagination
   (using `resilient_get`), **save a snapshot per non-empty page**, and return a DataFrame
   of all rows. Record each page's `source`, `params` (a copy), `status`, and `page` in the
   snapshot meta.


💡 **Hint.** `hashlib.sha256(payload_bytes).hexdigest()`; timestamp
`dt.datetime.now(dt.UTC).strftime("%Y%m%dT%H%M%SZ")`. In the loop: `resp =
resilient_get(...)`, `payload = resp.json()`, `rows = payload["data"]`; `if not rows: break`;
`page += 1`; `save_snapshot(resp.content, {...})`; extend; stop when `next_cursor is None`,
else set `params["cursor"]`. Use `params.copy()` in the meta so each line captures that
page's cursor.


In [ ]:
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def save_snapshot(payload_bytes, meta):
    # TODO: timestamp; write RAW_DIR/orders_<ts>_<page:05d>.json;
    #       append JSONL line to MANIFEST with timestamp, sha256, bytes; return the path
    raise NotImplementedError

def harvest_orders(base, headers, page_size=1000, **filters):
    url = f"{base}/v1/orders"
    params = {"limit": page_size, "cursor": 0, **filters}
    all_rows, page = [], 0
    # TODO: loop pages via cursor; save a snapshot per non-empty page; extend all_rows
    return pd.DataFrame(all_rows)

orders_df = harvest_orders(BASE_URL, HEADERS, page_size=1000)
print("harvested:", orders_df.shape)

In [ ]:
_snapshots = sorted(RAW_DIR.glob("orders_*.json"))
_manifest = [json.loads(line) for line in MANIFEST.open()]
check("A3: harvested all 10,000 orders", lambda: len(orders_df) == 10_000)
check("A3: one snapshot per page (10 pages of 1000)", lambda: len(_snapshots) == 10)
check("A3: manifest has one line per snapshot", lambda: len(_manifest) == len(_snapshots))
check("A3: every manifest record has a SHA-256 + byte size",
      lambda: all(len(r["sha256"]) == 64 and r["bytes"] > 0 for r in _manifest))
check("A3: manifest captures source + page provenance",
      lambda: all({"source", "params", "page", "timestamp"} <= set(r) for r in _manifest))

### A4 — Incremental harvest with a high-watermark
Re-harvesting everything every run is wasteful. A **high-watermark** stores the last date
you've seen; the next run asks only for rows **on or after** it. Implement the read/write
helpers and run an incremental harvest, then advance the watermark and prove a re-harvest
returns fewer rows.


💡 **Hint.** `read_watermark(default="2025-01-01")` returns the file's text if it exists
else `default`; `write_watermark(v)` writes it. Harvest with `since=<watermark>` (the API
filters `order_date >= since`). Advance to `inc["order_date"].max()`. A second harvest with
the advanced watermark returns only rows *on* that max date — strictly fewer.


In [ ]:
WATERMARK_FILE = ARTIFACTS / "last_watermark.txt"

def read_watermark(default="2025-01-01"):
    # TODO: return file text (stripped) if it exists, else default
    raise NotImplementedError

def write_watermark(value: str):
    # TODO: write value to WATERMARK_FILE
    raise NotImplementedError

start_wm = read_watermark()
inc = harvest_orders(BASE_URL, HEADERS, page_size=1000, since=start_wm)   # rows since watermark
new_wm = None       # TODO: advance to the max order_date observed, then write_watermark(new_wm)
inc2 = None         # TODO: re-harvest with since=new_wm
print("start watermark:", start_wm, "| new watermark:", new_wm,
      "| inc:", None if inc is None else len(inc), "| inc2:", None if inc2 is None else len(inc2))

In [ ]:
_con = sqlite3.connect(DB_PATH)
_sql_since = _con.execute("SELECT COUNT(*) FROM orders WHERE order_date >= ?", (start_wm,)).fetchone()[0]
_con.close()
check("A4: incremental harvest count matches SQL (since watermark)", lambda: len(inc) == _sql_since)
check("A4: watermark file advanced to the max observed date",
      lambda: WATERMARK_FILE.read_text().strip() == new_wm == max(str(d) for d in inc["order_date"]))
check("A4: re-harvest after advancing returns strictly fewer rows", lambda: len(inc2) < len(inc))
check("A4: those remaining rows are all on the watermark date",
      lambda: (inc2["order_date"] == new_wm).all())

### A5 — Normalize → partitioned Parquet
Write the harvested orders to Parquet **partitioned by region** — one file per
`store_region`, which lets a downstream reader skip regions it doesn't need.


💡 **Hint.** Select `["order_id","store_region","channel","order_date"]`, parse
`order_date` with `pd.to_datetime`. Then `for region, g in use.groupby("store_region"):`
write `pq.write_table(pa.Table.from_pandas(g, preserve_index=False), out_root /
f"store_region={region}.parquet")`.


In [ ]:
use = orders_df[["order_id", "store_region", "channel", "order_date"]].copy()
use["order_date"] = pd.to_datetime(use["order_date"])
out_root = PARQUET_DIR / "orders"
out_root.mkdir(parents=True, exist_ok=True)
# TODO: write one Parquet file per store_region into out_root
part_files = sorted(out_root.glob("store_region=*.parquet"))
print("partitions:", [p.name for p in part_files])

In [ ]:
_reload = pd.concat([pd.read_parquet(p) for p in part_files], ignore_index=True)
check("A5: one partition file per region (5)", lambda: len(part_files) == 5)
check("A5: partitions reload to the full row count", lambda: len(_reload) == len(orders_df))
check("A5: each partition holds exactly one region",
      lambda: all(pd.read_parquet(p)["store_region"].nunique() == 1 for p in part_files))

## Part B — SQL Extractor (parameterized · chunked)

Harvesting through the API is one path; reading the database directly is the other. Extract
the same kind of slice with SQL — safely (parameter binding) and scalably (chunked writes).


### B1 — Parameterized query with a join
Join `order_lines → orders → products`, filtered to a `region` and `since` date, with
**parameter binding** (`?` + `params=`), never string-formatting. Return: `order_id,
order_date, store_region, category, quantity, unit_price, discount_pct`.


💡 **Hint.** `pd.read_sql_query(sql, conn, params=(region, since))`. Join keys
`ol.order_id = o.order_id` and `ol.product_id = p.product_id`; `WHERE o.store_region = ?
AND o.order_date >= ?`.


In [ ]:
conn = sqlite3.connect(DB_PATH)
region, since = "West", "2025-01-01"
sql = """
-- TODO: SELECT the requested columns FROM order_lines ol
-- JOIN orders o ON ...  JOIN products p ON ...
-- WHERE o.store_region = ? AND o.order_date >= ?
"""
joined = None   # TODO: pd.read_sql_query(sql, conn, params=(region, since))
print(None if joined is None else joined.shape)

In [ ]:
check("B1: returned rows", lambda: len(joined) > 0)
check("B1: region filter held", lambda: (joined["store_region"] == "West").all())
check("B1: date filter held", lambda: (joined["order_date"] >= "2025-01-01").all())
check("B1: join populated product category", lambda: joined["category"].notna().all())

### B2 — Chunked reads → streaming Parquet
The full joined fact table is large. Stream it in **chunks** to a single Parquet file,
applying a per-chunk transform (parse `order_date`) so you never hold it all in memory.


💡 **Hint.** `pd.read_sql_query(sql_all, conn, chunksize=15_000)` yields DataFrames. For
each: parse dates, `pa.Table.from_pandas(chunk, preserve_index=False)`, open a
`pq.ParquetWriter(path, table.schema)` on the **first** chunk, `write_table` each, and
`close()` at the end.


In [ ]:
sql_all = """
SELECT o.order_id, o.order_date, o.store_region,
       p.category, ol.quantity, ol.unit_price, ol.discount_pct
FROM order_lines ol
JOIN orders   o ON ol.order_id = o.order_id
JOIN products p ON ol.product_id = p.product_id
"""
OUT = PARQUET_DIR / "order_lines_joined.parquet"
writer, rows_written = None, 0
# TODO: stream chunks (size 15000), parse order_date per chunk, write to one Parquet file
print("rows written:", rows_written)

In [ ]:
_con = sqlite3.connect(DB_PATH)
_lines = _con.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0]
_con.close()
_back = pd.read_parquet(OUT)
check("B2: streamed every joined line", lambda: rows_written == _lines)
check("B2: Parquet round-trips the row count", lambda: len(_back) == _lines)
check("B2: order_date persisted as datetime", lambda: str(_back["order_date"].dtype).startswith("datetime64"))

## Part C — Stretch: data contract · idempotency · retry budget

*(For fast finishers — each is independent.)* These harden the pipeline: validate before
you trust, make re-runs safe, and bound worst-case retry time.

> **Responsible-AI tie-in.** A **data contract** is the quality gate that keeps malformed
> rows out of an LLM's training/grounding set; **provenance + idempotency** are what let you
> *audit and reproduce* exactly what data shaped a model.


### C1 — Enforce a data contract with pandera
Define an `OrdersSchema` and validate the harvested `orders_df`. Then prove it **rejects**
bad data.

> **⚠️ Currency:** import pandera via `import pandera.pandas as pa` — the older
> `import pandera as pa` now emits a `FutureWarning` (pandera ≥ 0.20).


💡 **Hint.** `DataFrameSchema({...}, strict=False)` so extra columns are allowed.
`Column(int, Check.gt(0), unique=True)` for `order_id`; `Column(str,
Check.isin(REGIONS))` for `store_region`. Validate with `OrdersSchema.validate(orders_df)`;
a bad frame raises `pa.errors.SchemaError`.


In [ ]:
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

REGIONS = ["Southeast", "Northeast", "Midwest", "West", "Southwest"]
OrdersSchema = None   # TODO: DataFrameSchema for order_id (int>0, unique), store_region
                      #       (str, isin REGIONS), channel (str), order_date (str); strict=False

validated = None      # TODO: OrdersSchema.validate(orders_df)

def rejects_bad_region():
    bad = orders_df.copy()
    bad.loc[bad.index[0], "store_region"] = "Atlantis"
    try:
        OrdersSchema.validate(bad); return False
    except pa.errors.SchemaError:
        return True

In [ ]:
check("C1: contract validates the harvested orders", lambda: len(validated) == len(orders_df))
check("C1: contract rejects an out-of-vocabulary region", rejects_bad_region)

### C2 — Idempotent re-runs (upsert by key)
A harvester that's run twice must not double its data. Implement `upsert_orders` that merges
new rows into a master Parquet, **replacing by `order_id`**, then show a second identical run
leaves the row count unchanged.


💡 **Hint.** Load the master if it exists. `combined =
pd.concat([base[~base["order_id"].isin(new["order_id"])], new])` drops the old versions of
any incoming keys, then appends the new. Write back with `to_parquet(index=False)`.


In [ ]:
MASTER = PARQUET_DIR / "orders_master.parquet"

def upsert_orders(new_df, master_path=MASTER):
    cols = ["order_id", "store_region", "channel", "order_date"]
    new = new_df[cols].copy()
    # TODO: if master exists, drop rows whose order_id is in `new`, concat new, else use new
    # TODO: sort by order_id, write Parquet (index=False), return combined
    raise NotImplementedError

first = upsert_orders(orders_df)
second = upsert_orders(orders_df)   # identical re-run
print("after first:", len(first), "| after second:", len(second))

In [ ]:
check("C2: first upsert loads all 10,000", lambda: len(first) == 10_000)
check("C2: identical re-run does NOT duplicate (idempotent)", lambda: len(second) == 10_000)
check("C2: order_id remains unique in the master", lambda: second["order_id"].is_unique)

### C3 — Bound the worst case with a retry budget
Exponential backoff can, in the limit, sleep for a very long time. Add a **total sleep
budget**: `resilient_get_budget(url, ..., max_sleep=1.0)` aborts with `TimeoutError` once
cumulative sleep exceeds the budget.


💡 **Hint.** Track `slept`. After each `time.sleep(nap)`, add `nap` to `slept`; if `slept >
max_sleep`, `raise TimeoutError(...)`. Point it at `/v1/unreliable` with a high `fail_times`
(always fails) to trip the budget fast; point it at `/v1/orders` to confirm success returns.


In [ ]:
def resilient_get_budget(url, params=None, headers=None, max_retries=10, max_sleep=1.0):
    slept, backoff = 0.0, 0.2
    for _ in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
        except (requests.Timeout, requests.ConnectionError):
            delay = backoff
        else:
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                delay = float(r.headers.get("Retry-After", backoff))
            elif r.status_code in (500, 502, 503, 504):
                delay = backoff
            else:
                raise HarvestError(f"Non-retryable {r.status_code}")
        # TODO: nap = delay + jitter; sleep; slept += nap;
        #       if slept > max_sleep: raise TimeoutError(...); backoff = min(backoff*2, 8)
        raise NotImplementedError
    raise TimeoutError("exhausted retries")

def budget_trips():
    requests.post(f"{BASE_URL}/admin/reset", timeout=5)
    try:
        resilient_get_budget(f"{BASE_URL}/v1/unreliable",
                             {"key": "c3", "fail_times": 100}, HEADERS, max_sleep=1.0)
        return False
    except TimeoutError:
        return True

In [ ]:
check("C3: a healthy call succeeds within budget",
      lambda: resilient_get_budget(f"{BASE_URL}/v1/orders", {"limit": 1}, HEADERS).status_code == 200)
check("C3: persistent failure trips the sleep budget (TimeoutError)", budget_trips)

## Wrap-up

You built a harvester you can **defend**: authenticated, resilient (backoff + jitter +
`Retry-After`), paginated, and — critically — **provenance-tracked** (raw snapshots + a
checksummed manifest + a watermark), plus a SQL extractor and hardening (contract,
idempotency, retry budget). Answer in a markdown cell:

1. How does your harvester treat `429` vs `500` vs `401`? What does `Retry-After` convey, and why add **jitter**?
2. Explain the **high-watermark** logic. What breaks if data arrives out of order, and how would you mitigate it?
3. Why are **parameterized** queries non-negotiable? Give a one-line risky example.
4. Which single artifact would you hand an auditor to prove *what* data you ingested and *that it's intact*? Why?

### Further exploration *(pointers — not built out here)*
- **Conditional requests (ETags / `If-None-Match` / `304`)** — skip re-downloading unchanged pages.
- **Bounded concurrency** — a client-side **token bucket** + `ThreadPoolExecutor` to fetch pages in parallel while respecting the rate limit.
- **Structured logging** — one JSON log line per page (attempts, sleep, rows, elapsed).
- **CLI packaging** — wrap the harvester in a `typer`/`argparse` CLI driven by a `harvest.yml`.
- **Unit tests** — `pytest` + `responses` to simulate `503→200` and watermark advancement offline (as in Lab 01's Part-3 style).


In [ ]:
conn.close()
server.should_exit = True
time.sleep(0.3)
print("Local API stopped.")
print("Provenance written:", len(list(RAW_DIR.glob('*.json'))), "snapshots +",
      sum(1 for _ in MANIFEST.open()), "manifest lines")